In [1]:
from modeling import *

In [2]:
device = torch.device("cpu")
config = config = BertConfig.from_json_file('/Users/saptarshimallikthakur/Downloads/uncased_L-4_H-256_A-4/bert_config.json')
model = BertModel(config)
model.load_state_dict(torch.load('/Users/saptarshimallikthakur/Pictures/VLM/bert/pytorch_model.bin', map_location='cpu'))
model.to(device)
for p in model.parameters():
    p.requires_grad = False

In [3]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")


Total parameters     : 11,170,560
Trainable parameters : 0


In [4]:
def load_vocab(vocab_file):
    vocab = {}
    with open(vocab_file, "r") as f:
        for idx, token in enumerate(f):
            vocab[token.strip()] = idx
    return vocab

vocab = load_vocab("/Users/saptarshimallikthakur/Downloads/uncased_L-4_H-256_A-4/vocab.txt")

inv_vocab = {v: k for k, v in vocab.items()}

In [5]:
import re

def basic_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()

def wordpiece_tokenize(word, vocab, unk_token="[UNK]"):
    if word in vocab:
        return [word]

    tokens = []
    start = 0
    while start < len(word):
        end = len(word)
        cur_substr = None
        while start < end:
            substr = word[start:end]
            if start > 0:
                substr = "##" + substr
            if substr in vocab:
                cur_substr = substr
                break
            end -= 1
        if cur_substr is None:
            return [unk_token]
        tokens.append(cur_substr)
        start = end
    return tokens

def tokenize(text, vocab):
    tokens = []
    for word in basic_tokenize(text):
        tokens.extend(wordpiece_tokenize(word, vocab))
    return tokens


def encode(text, vocab, max_len=32):
    tokens = tokenize(text, vocab)
    tokens = ["[CLS]"] + tokens + ["[SEP]"]

    input_ids = [vocab.get(t, vocab["[UNK]"]) for t in tokens]

    if len(input_ids) < max_len:
        input_ids += [vocab["[PAD]"]] * (max_len - len(input_ids))
    else:
        input_ids = input_ids[:max_len]

    attention_mask = [1 if i != vocab["[PAD]"] else 0 for i in input_ids]

    return torch.tensor([input_ids]), torch.tensor([attention_mask]), tokens

In [6]:
model.eval()

text = "a cat sitting on the mat"

input_ids, attention_mask, tokens = encode(text, vocab)
input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

with torch.no_grad():
    outputs = model(input_ids, attention_mask=attention_mask)


In [7]:
input_ids

tensor([[  101,  1037,  4937,  3564,  2006,  1996, 13523,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0]])

In [8]:
outputs[0]

[tensor([[[ 0.4660, -0.2510,  0.6602,  ..., -0.0208, -0.5053,  0.5695],
          [ 0.4817,  0.3483, -0.5027,  ...,  0.5170, -0.6487, -0.5480],
          [-0.7244,  0.9604, -1.2871,  ..., -1.3513,  0.2463, -1.6804],
          ...,
          [-0.2242, -0.0665, -0.6112,  ..., -0.3057, -0.2413, -0.2974],
          [-0.2345,  0.1866, -0.5474,  ..., -0.3709,  0.1208, -0.4989],
          [-0.2229,  0.2955, -0.5155,  ..., -0.3708,  0.2726, -0.6638]]]),
 tensor([[[ 0.1794, -0.3303,  0.4454,  ..., -0.1362, -0.7521,  0.1382],
          [-0.1317,  0.6265, -0.4252,  ...,  0.1660, -0.9270, -0.6955],
          [-0.6709,  0.8564, -0.5091,  ..., -1.4113,  0.0625, -1.7707],
          ...,
          [-0.8612,  0.0387, -0.4953,  ..., -0.7115, -0.6590, -0.0569],
          [-0.8542,  0.3794, -0.2872,  ..., -0.7963, -0.4195, -0.2278],
          [-0.6855,  0.3012, -0.1353,  ..., -0.8173, -0.2160, -0.5780]]]),
 tensor([[[ 0.1526, -0.3685,  0.4067,  ...,  0.3263, -0.5945,  0.2907],
          [-0.0482,  0.5089,

In [9]:
outputs[0][-1].shape

torch.Size([1, 32, 256])

In [10]:
import torch
import torch.nn.functional as F

def get_sentence_embedding(
    text,
    model,
    vocab,
    device,
    max_len=32
):
    input_ids, attention_mask, _ = encode(text, vocab, max_len)
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)

    last_hidden = outputs[0][-1]  # (1, seq_len, hidden)

    # Mean pooling
    mask = attention_mask.unsqueeze(-1).float()
    emb = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1)

    return emb.squeeze(0)  # (hidden_size,)

def cosine_similarity(sent1, sent2):
    emb1 = get_sentence_embedding(sent1, model, vocab, device)
    emb2 = get_sentence_embedding(sent2, model, vocab, device)

    sim = F.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0))
    return sim.item()



In [11]:
s1 = "a cat sitting on a mat"
s2 = "ocean full of fish"
s3 = "a cat resting on a mat"

print(cosine_similarity(s1, s2))  # lower
print(cosine_similarity(s1, s3))  # higher


0.5042646527290344
0.9352786540985107
